<a href="https://colab.research.google.com/github/yukeshanand/masai_aiml_task/blob/main/python_task1_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# task1_data_collection.py

import requests
import json
import os
import time
from datetime import datetime

BASE_URL = "https://hacker-news.firebaseio.com/v0"
HEADERS = {"User-Agent": "TrendPulse/1.0"}

# Category keywords
CATEGORIES = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

def get_category(title):
    title = title.lower()
    for cat, keywords in CATEGORIES.items():
        if any(k in title for k in keywords):
            return cat
    return None

def fetch_json(url):
    try:
        res = requests.get(url, headers=HEADERS, timeout=5)
        return res.json()
    except:
        print(f"Failed: {url}")
        return None

def main():
    ids = fetch_json(f"{BASE_URL}/topstories.json")[:500]
    data = []
    count = {k: 0 for k in CATEGORIES}

    for cat in CATEGORIES:
        for i in ids:
            if count[cat] >= 25:
                break

            story = fetch_json(f"{BASE_URL}/item/{i}.json")
            if not story or "title" not in story:
                continue

            assigned = get_category(story["title"])
            if assigned == cat:
                data.append({
                    "post_id": story.get("id"),
                    "title": story.get("title"),
                    "category": cat,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0),
                    "author": story.get("by"),
                    "collected_at": datetime.now().isoformat()
                })
                count[cat] += 1

        time.sleep(2)  # one sleep per category

    # Save file
    os.makedirs("data", exist_ok=True)
    filename = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"

    with open(filename, "w") as f:
        json.dump(data, f, indent=4)

    print(f"Collected {len(data)} stories. Saved to {filename}")

if __name__ == "__main__":
    main()

Collected 78 stories. Saved to data/trends_20260405.json


In [ ]:
# task2_data_processing.py

import pandas as pd
import os

# Load JSON file (latest file in data folder)
files = [f for f in os.listdir("data") if f.startswith("trends_") and f.endswith(".json")]
files.sort()
filepath = os.path.join("data", files[-1])

df = pd.read_json(filepath)
print(f"Loaded {len(df)} stories from {filepath}")

# --- Cleaning ---

# 1. Remove duplicates
df = df.drop_duplicates(subset="post_id")
print(f"After removing duplicates: {len(df)}")

# 2. Remove missing values
df = df.dropna(subset=["post_id", "title", "score"])
print(f"After removing nulls: {len(df)}")

# 3. Fix data types
df["score"] = df["score"].astype(int)
df["num_comments"] = df["num_comments"].fillna(0).astype(int)

# 4. Remove low-quality stories
df = df[df["score"] >= 5]
print(f"After removing low scores: {len(df)}")

# 5. Clean whitespace
df["title"] = df["title"].str.strip()

# --- Save CSV ---
os.makedirs("data", exist_ok=True)
csv_path = "data/trends_clean.csv"
df.to_csv(csv_path, index=False)

print(f"\nSaved {len(df)} rows to {csv_path}")

# --- Summary ---
print("\nStories per category:")
print(df["category"].value_counts())

Loaded 78 stories from data/trends_20260405.json
After removing duplicates: 78
After removing nulls: 78
After removing low scores: 74

Saved 74 rows to data/trends_clean.csv

Stories per category:
category
technology       24
entertainment    23
worldnews        11
science          10
sports            6
Name: count, dtype: int64
